# Machine Learning for Radiomics Classification

**Objective:** Train and evaluate machine learning models for cancer classification using radiomics features

**Learning Outcomes:**
1. Data preprocessing and feature standardization
2. Feature selection techniques
3. Cross-validation strategy
4. Model training (Logistic Regression, Random Forest, SVM)
5. Performance evaluation metrics
6. Model interpretation

**Workflow:**
```
Raw Features → Preprocessing → Feature Selection → Model Training → Evaluation → Interpretation
```

## 1. Setup & Imports

In [ ]:
# Install required libraries
!pip install -q pandas numpy matplotlib seaborn scikit-learn scipy

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.pipeline import Pipeline  # Prevents data leakage!
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# Set plotting style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')  # Fallback for older matplotlib
sns.set_palette("husl")

print("✓ Libraries imported successfully")

### Learning Curves

Learning curves show whether a model benefits from **more data** or is **overfitting**.
- If train score >> test score → **overfitting** (try more data or simpler model)
- If both scores are low → **underfitting** (try more features or complex model)
- If scores converge → **good fit**

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curves(models, X, y, cv=5, figsize=(16, 5)):
    """
    Plot learning curves for multiple models.
    Shows how performance changes with increasing training data.
    """
    fig, axes = plt.subplots(1, len(models), figsize=figsize)
    if len(models) == 1:
        axes = [axes]
    
    train_sizes = np.linspace(0.2, 1.0, 8)
    
    for ax, (name, model) in zip(axes, models.items()):
        sizes, train_scores, test_scores = learning_curve(
            model, X, y, cv=cv, n_jobs=-1,
            train_sizes=train_sizes, scoring='accuracy'
        )
        
        train_mean = train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        test_mean = test_scores.mean(axis=1)
        test_std = test_scores.std(axis=1)
        
        ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#2E86AB')
        ax.fill_between(sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='#A23B72')
        ax.plot(sizes, train_mean, 'o-', color='#2E86AB', label='Training score')
        ax.plot(sizes, test_mean, 'o-', color='#A23B72', label='Cross-validation score')
        
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel('Training Set Size')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0.4, 1.05)
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Annotate overfitting gap
        gap = train_mean[-1] - test_mean[-1]
        if gap > 0.1:
            ax.annotate(f'Gap: {gap:.2f}\n(overfitting)', xy=(sizes[-1], test_mean[-1]),
                       fontsize=8, color='red', ha='right')
    
    plt.suptitle('Learning Curves — Does More Data Help?', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/learning_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot learning curves for our models
models_lc = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42)
}

plot_learning_curves(models_lc, X_train_selected, y_train)
print('\n📈 Interpretation guide:')
print('  • Large gap between curves → overfitting (try regularization or more data)')
print('  • Both curves low → underfitting (try more features or complex model)')
print('  • Curves converge → good model capacity for this data')

## 2. Load Data

Load the features extracted from the previous notebook

In [ ]:
# Load features
# If running from scratch, generate synthetic data
try:
    features_df = pd.read_csv('../outputs/radiomics_features.csv')
    print("✓ Loaded extracted features")
except FileNotFoundError:
    print("⚠ Feature file not found. Generating realistic synthetic data...")
    
    np.random.seed(42)
    n_samples = 100
    n_benign = n_samples // 2
    n_malignant = n_samples - n_benign
    
    # Define realistic radiomics feature names (50+ features)
    # Format: feature_name: (benign_mean, benign_std, malignant_mean, malignant_std)
    feature_defs = {
        # First-order statistics (18 features)
        'original_firstorder_Mean': (100, 20, 150, 30),
        'original_firstorder_Median': (95, 18, 145, 28),
        'original_firstorder_Variance': (500, 100, 800, 200),
        'original_firstorder_Skewness': (0.2, 0.3, 0.8, 0.4),
        'original_firstorder_Kurtosis': (3.0, 0.5, 4.5, 0.8),
        'original_firstorder_Energy': (10000, 2000, 20000, 4000),
        'original_firstorder_Entropy': (3.5, 0.5, 4.5, 0.7),
        'original_firstorder_Minimum': (-50, 20, -80, 30),
        'original_firstorder_Maximum': (300, 50, 450, 80),
        'original_firstorder_Range': (350, 60, 530, 90),
        'original_firstorder_MeanAbsoluteDeviation': (20, 5, 35, 8),
        'original_firstorder_RobustMeanAbsoluteDeviation': (15, 4, 25, 6),
        'original_firstorder_RootMeanSquared': (110, 22, 165, 33),
        'original_firstorder_Uniformity': (0.05, 0.01, 0.03, 0.008),
        'original_firstorder_10Percentile': (50, 15, 70, 20),
        'original_firstorder_90Percentile': (160, 25, 240, 35),
        'original_firstorder_InterquartileRange': (40, 10, 65, 15),
        'original_firstorder_TotalEnergy': (50000, 10000, 100000, 20000),
        # Shape features (14 features)
        'original_shape_VoxelVolume': (1000, 200, 2000, 400),
        'original_shape_MeshVolume': (980, 195, 1960, 390),
        'original_shape_SurfaceArea': (500, 80, 800, 150),
        'original_shape_SurfaceVolumeRatio': (0.5, 0.1, 0.4, 0.08),
        'original_shape_Sphericity': (0.8, 0.1, 0.6, 0.15),
        'original_shape_Compactness1': (0.3, 0.05, 0.2, 0.04),
        'original_shape_Compactness2': (0.25, 0.04, 0.18, 0.035),
        'original_shape_Maximum3DDiameter': (25, 5, 40, 8),
        'original_shape_Maximum2DDiameterSlice': (20, 4, 32, 6),
        'original_shape_Maximum2DDiameterColumn': (18, 3.5, 28, 5.5),
        'original_shape_Maximum2DDiameterRow': (17, 3.2, 27, 5.2),
        'original_shape_MajorAxisLength': (22, 4, 35, 7),
        'original_shape_MinorAxisLength': (15, 3, 22, 4.5),
        'original_shape_Elongation': (0.7, 0.1, 0.55, 0.12),
        'original_shape_Flatness': (0.6, 0.08, 0.45, 0.1),
        # GLCM texture features (10 features)
        'original_glcm_Contrast': (50, 10, 100, 20),
        'original_glcm_Correlation': (0.7, 0.1, 0.5, 0.15),
        'original_glcm_JointEnergy': (0.02, 0.005, 0.01, 0.003),
        'original_glcm_JointEntropy': (6.0, 0.5, 7.5, 0.8),
        'original_glcm_Homogeneity': (0.3, 0.05, 0.2, 0.04),
        'original_glcm_DifferenceVariance': (15, 3, 25, 5),
        'original_glcm_DifferenceEntropy': (2.5, 0.4, 3.5, 0.6),
        'original_glcm_SumEntropy': (4.0, 0.5, 5.5, 0.7),
        'original_glcm_MaximumProbability': (0.08, 0.02, 0.05, 0.015),
        'original_glcm_Imc1': (-0.1, 0.05, -0.15, 0.06),
        # GLRLM texture features (6 features)
        'original_glrlm_GrayLevelNonUniformity': (100, 20, 200, 40),
        'original_glrlm_RunLengthNonUniformity': (500, 80, 800, 120),
        'original_glrlm_ShortRunEmphasis': (0.85, 0.05, 0.75, 0.08),
        'original_glrlm_LongRunEmphasis': (1.5, 0.2, 2.0, 0.3),
        'original_glrlm_RunPercentage': (0.9, 0.03, 0.8, 0.05),
        'original_glrlm_RunEntropy': (4.0, 0.5, 5.0, 0.6),
        # GLSZM texture features (6 features)
        'original_glszm_SizeZoneNonUniformity': (200, 40, 400, 80),
        'original_glszm_SmallAreaEmphasis': (0.6, 0.1, 0.4, 0.12),
        'original_glszm_LargeAreaEmphasis': (50, 15, 100, 30),
        'original_glszm_ZoneEntropy': (5.0, 0.6, 6.5, 0.8),
        'original_glszm_GrayLevelVariance': (25, 5, 40, 8),
        'original_glszm_ZoneVariance': (30, 8, 55, 12),
    }
    
    # Generate data with class-specific distributions
    data_rows = []
    for i in range(n_samples):
        is_malignant = i >= n_benign
        row = {'patient_id': f'Patient_{i+1:03d}', 'label': int(is_malignant)}
        for feat, (b_mean, b_std, m_mean, m_std) in feature_defs.items():
            if is_malignant:
                row[feat] = np.random.normal(m_mean, m_std)
            else:
                row[feat] = np.random.normal(b_mean, b_std)
        data_rows.append(row)
    
    features_df = pd.DataFrame(data_rows)
    
    # Save
    os.makedirs('../outputs', exist_ok=True)
    features_df.to_csv('../outputs/radiomics_features.csv', index=False)
    print(f"✓ Generated {len(feature_defs)} realistic radiomics features")

print(f"\nDataset shape: {features_df.shape}")
print(f"\nClass distribution:")
print(features_df['label'].value_counts())
features_df.head()

## 3. Data Preprocessing

### 3.1 Handle Missing Values

In [ ]:
# Check for missing values
feature_cols = [col for col in features_df.columns 
                if col not in ['patient_id', 'label']]

print(f"Total features: {len(feature_cols)}")
print(f"\nMissing values per feature:")
missing = features_df[feature_cols].isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  No missing values found ✓")

# Handle missing values (if any)
if missing.sum() > 0:
    features_df[feature_cols] = features_df[feature_cols].fillna(
        features_df[feature_cols].median()
    )
    print("\n✓ Filled missing values with median")

# Check for infinite values
inf_counts = np.isinf(features_df[feature_cols]).sum().sum()
print(f"\nInfinite values: {inf_counts}")
if inf_counts > 0:
    features_df[feature_cols] = features_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    features_df[feature_cols] = features_df[feature_cols].fillna(features_df[feature_cols].median())
    print("✓ Replaced infinite values")

### 3.2 Feature Standardization

Radiomics features have different scales - standardization is essential!

In [ ]:
# Prepare data
X = features_df[feature_cols].values
y = features_df['label'].values
patient_ids = features_df['patient_id'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

# Split data (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTrain class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features standardized")
print(f"Train mean (should be ~0): {X_train_scaled.mean(axis=0).mean():.4f}")
print(f"Train std (should be ~1): {X_train_scaled.std(axis=0).mean():.4f}")

### 3.3 Feature Distribution Visualization

In [ ]:
# Visualize feature distributions before and after scaling
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Before scaling
axes[0, 0].boxplot(X_train[:, :5])
axes[0, 0].set_title('Before Scaling (First 5 Features)')
axes[0, 0].set_ylabel('Value')

# After scaling
axes[0, 1].boxplot(X_train_scaled[:, :5])
axes[0, 1].set_title('After Scaling (First 5 Features)')
axes[0, 1].set_ylabel('Standardized Value')

# Class distribution
axes[1, 0].bar(['Benign', 'Malignant'], np.bincount(y))
axes[1, 0].set_title('Class Distribution')
axes[1, 0].set_ylabel('Count')

# Feature correlation (subset)
if len(feature_cols) >= 5:
    corr = pd.DataFrame(X_train_scaled, columns=feature_cols).iloc[:, :5].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
    axes[1, 1].set_title('Feature Correlation (First 5)')

plt.tight_layout()
plt.savefig('../outputs/preprocessing_overview.png', dpi=150)
plt.show()

## 4. Feature Selection

Reduce dimensionality and improve model performance

In [ ]:
# Step 1: Remove highly correlated features (|r| > 0.9)
corr_matrix = pd.DataFrame(X_train_scaled, columns=feature_cols).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

if len(to_drop) > 0:
    drop_idx = [feature_cols.index(c) for c in to_drop]
    keep_idx = [i for i in range(len(feature_cols)) if i not in drop_idx]
    X_train_decorr = X_train_scaled[:, keep_idx]
    X_test_decorr = X_test_scaled[:, keep_idx]
    feature_cols_decorr = [feature_cols[i] for i in keep_idx]
    print(f"Removed {len(to_drop)} highly correlated features (|r| > 0.9):")
    for feat in to_drop:
        print(f"  - {feat}")
else:
    X_train_decorr = X_train_scaled
    X_test_decorr = X_test_scaled
    feature_cols_decorr = feature_cols.copy()
    print("No highly correlated feature pairs found (|r| > 0.9)")

print(f"\nFeatures: {len(feature_cols)} → {len(feature_cols_decorr)} after correlation filter")

# Step 2: Compare feature selection methods

# Method 1: Statistical Test (F-test)
selector_f = SelectKBest(f_classif, k=min(10, len(feature_cols_decorr)))
X_train_f = selector_f.fit_transform(X_train_decorr, y_train)
f_scores = selector_f.scores_
f_selected = selector_f.get_support()

# Method 2: Mutual Information
selector_mi = SelectKBest(mutual_info_classif, k=min(10, len(feature_cols_decorr)))
X_train_mi = selector_mi.fit_transform(X_train_decorr, y_train)
mi_scores = selector_mi.scores_
mi_selected = selector_mi.get_support()

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# F-test scores
top_k = min(15, len(feature_cols_decorr))
top_f_idx = np.argsort(f_scores)[-top_k:]
axes[0].barh(range(top_k), f_scores[top_f_idx])
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels([feature_cols_decorr[i][:30] for i in top_f_idx], fontsize=8)
axes[0].set_xlabel('F-Score')
axes[0].set_title(f'Top {top_k} Features (F-Test)')

# Mutual Information scores
top_mi_idx = np.argsort(mi_scores)[-top_k:]
axes[1].barh(range(top_k), mi_scores[top_mi_idx])
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([feature_cols_decorr[i][:30] for i in top_mi_idx], fontsize=8)
axes[1].set_xlabel('Mutual Information')
axes[1].set_title(f'Top {top_k} Features (Mutual Information)')

plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150)
plt.show()

print("\n✓ Feature selection completed")
print(f"\nTop 5 features (F-test):")
for i, idx in enumerate(top_f_idx[-5:]):
    print(f"  {i+1}. {feature_cols_decorr[idx]}: {f_scores[idx]:.2f}")

In [ ]:
# Use selected features for modeling
# Apply F-test on decorrelated features
n_selected = min(10, len(feature_cols_decorr))
selector = SelectKBest(f_classif, k=n_selected)
X_train_selected = selector.fit_transform(X_train_decorr, y_train)
X_test_selected = selector.transform(X_test_decorr)

selected_features = [feature_cols_decorr[i] for i in selector.get_support(indices=True)]

print(f"Selected {n_selected} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

print(f"\nFeature matrix shape: {X_train_selected.shape}")

## 5. Model Training with Cross-Validation

### 5.1 Define Models and Cross-Validation Strategy

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42)
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Models to evaluate:")
for name in models.keys():
    print(f"  - {name}")
print(f"\nCross-validation: {cv.n_splits}-fold stratified")

In [ ]:
# Evaluate models using cross-validation
cv_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='roc_auc')
    
    cv_results[name] = {
        'mean_auc': scores.mean(),
        'std_auc': scores.std(),
        'scores': scores
    }
    
    print(f"  CV AUC: {scores.mean():.3f} (+/- {scores.std()*2:.3f})")

# Find best model
best_model_name = max(cv_results.keys(), 
                     key=lambda k: cv_results[k]['mean_auc'])
print(f"\n✓ Best model: {best_model_name}")
print(f"  AUC: {cv_results[best_model_name]['mean_auc']:.3f}")

In [ ]:
# Visualize cross-validation results
fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(cv_results.keys())
means = [cv_results[m]['mean_auc'] for m in model_names]
stds = [cv_results[m]['std_auc'] for m in model_names]

x = np.arange(len(model_names))
ax.bar(x, means, yerr=stds, capsize=5, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.set_ylabel('AUC Score')
ax.set_title('Cross-Validation AUC Scores')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='r', linestyle='--', label='Random')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/cv_comparison.png', dpi=150)
plt.show()

### 5.2 Hyperparameter Tuning (Optional)

In [ ]:
# Optional: Grid search for best model
if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7, None],
        'min_samples_split': [2, 5]
    }
    model = RandomForestClassifier(random_state=42)
elif 'SVM' in best_model_name:
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.01, 0.001]
    }
    model = SVC(kernel='rbf', probability=True, random_state=42)
else:
    param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
    model = LogisticRegression(max_iter=1000, random_state=42)

print(f"Tuning {best_model_name}...")
grid_search = GridSearchCV(
    model, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train_selected, y_train)

print(f"\n✓ Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.3f}")

# Use best model
best_model = grid_search.best_estimator_

## 6. Final Model Evaluation

### 6.1 Train and Evaluate on Test Set

In [ ]:
# Train final model on full training set
best_model.fit(X_train_selected, y_train)

# Predict on test set
y_pred = best_model.predict(X_test_selected)
y_pred_proba = best_model.predict_proba(X_test_selected)[:, 1]

# Calculate metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'AUC': roc_auc_score(y_test, y_pred_proba)
}

print("\n=== Test Set Performance ===")
for metric, value in metrics.items():
    print(f"{metric}: {value:.3f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, 
                           target_names=['Benign', 'Malignant']))

### 6.2 Confusion Matrix and ROC Curve

In [ ]:
# Create comprehensive evaluation plots
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')
axes[0, 0].set_xticklabels(['Benign', 'Malignant'])
axes[0, 0].set_yticklabels(['Benign', 'Malignant'])

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, 'b-', linewidth=2, 
                label=f'ROC (AUC = {metrics["AUC"]:.3f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].legend()
axes[0, 1].grid(True)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1, 0].plot(recall, precision, 'g-', linewidth=2)
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].grid(True)

# 4. Prediction Distribution
axes[1, 1].hist(y_pred_proba[y_test == 0], alpha=0.5, bins=20, 
                label='Benign', color='blue')
axes[1, 1].hist(y_pred_proba[y_test == 1], alpha=0.5, bins=20, 
                label='Malignant', color='red')
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[1, 1].set_xlabel('Predicted Probability')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Prediction Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../outputs/model_evaluation.png', dpi=150)
plt.show()

## 7. Model Interpretation

Understand which features drive predictions

In [ ]:
# Feature importance (for applicable models)
if hasattr(best_model, 'feature_importances_'):
    # Tree-based models
    importances = best_model.feature_importances_
    importance_type = 'Feature Importance'
elif hasattr(best_model, 'coef_'):
    # Linear models
    importances = np.abs(best_model.coef_[0])
    importance_type = 'Coefficient Magnitude'
else:
    importances = None
    importance_type = None

if importances is not None:
    # Sort by importance
    indices = np.argsort(importances)[::-1]
    
    # Plot
    plt.figure(figsize=(10, 6))
    n_show = min(10, len(importances))
    plt.barh(range(n_show), importances[indices[:n_show]])
    plt.yticks(range(n_show), 
               [selected_features[i][:40] for i in indices[:n_show]], 
               fontsize=9)
    plt.xlabel(importance_type)
    plt.title(f'Top {n_show} Most Important Features')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance_model.png', dpi=150)
    plt.show()
    
    print(f"\nTop 5 most important features:")
    for i in range(min(5, len(importances))):
        idx = indices[i]
        print(f"  {i+1}. {selected_features[idx]}: {importances[idx]:.4f}")
else:
    print("Feature importance not available for this model type")

## 8. Summary and Best Practices

In [ ]:
# Create summary report
summary = f"""

# Radiomics Machine Learning Summary

## Dataset
- Total samples: {len(features_df)}
- Features: {len(feature_cols)}
- Selected features: {n_selected}
- Classes: {dict(zip(*np.unique(y, return_counts=True)))}

## Preprocessing
- Standardization: StandardScaler
- Feature selection: F-test (SelectKBest)
- Missing values: Median imputation

## Cross-Validation Results
"""

for name, results in cv_results.items():
    summary += f"- {name}: {results['mean_auc']:.3f} (+/- {results['std_auc']*2:.3f})\n"

summary += f"""
## Final Test Results
- Best model: {best_model_name}
- Test Accuracy: {metrics['Accuracy']:.3f}
- Test AUC: {metrics['AUC']:.3f}
- Test Precision: {metrics['Precision']:.3f}
- Test Recall: {metrics['Recall']:.3f}
- Test F1: {metrics['F1-Score']:.3f}

## Key Findings
"""

if importances is not None:
    top_3_idx = np.argsort(importances)[-3:][::-1]
    summary += "Top 3 predictive features:\n"
    for i, idx in enumerate(top_3_idx, 1):
        summary += f"{i}. {selected_features[idx]}\n"

summary += """

## Recommendations
1. Feature standardization is crucial for SVM and Logistic Regression
2. Feature selection reduces overfitting and improves interpretability
3. Cross-validation provides robust estimate of model performance
4. Always evaluate on held-out test set

## Next Steps
1. Collect more data to improve generalization
2. Try ensemble methods (e.g., VotingClassifier)
3. Investigate feature engineering (e.g., ratios, interactions)
4. Validate on external dataset
"""

print(summary)

# Save summary
with open('../outputs/summary_report.txt', 'w') as f:
    f.write(summary)
print("\n✓ Summary saved to ../outputs/summary_report.txt")

## Appendix: Complete ML Pipeline Function

For easy reuse, here's the complete pipeline in one function:

In [ ]:
def radiomics_ml_pipeline(features_df, feature_cols, label_col='label', 
                       n_features=10, test_size=0.2, random_state=42):
    """
    Complete radiomics machine learning pipeline
    
    Args:
        features_df: DataFrame with features and labels
        feature_cols: List of feature column names
        label_col: Name of label column
        n_features: Number of features to select
        test_size: Fraction for test set
        random_state: Random seed
    
    Returns:
        Dictionary with results and trained model
    """
    # Prepare data
    X = features_df[feature_cols].values
    y = features_df[label_col].values
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Scale
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # Select features
    selector = SelectKBest(f_classif, k=n_features)
    X_train_sel = selector.fit_transform(X_train_s, y_train)
    X_test_sel = selector.transform(X_test_s)
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=random_state)
    model.fit(X_train_sel, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test_sel)
    y_proba = model.predict_proba(X_test_sel)[:, 1]
    
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_proba),
        'model': model,
        'scaler': scaler,
        'selector': selector,
        'test_predictions': y_pred
    }
    
    return results

# Example usage:
# results = radiomics_ml_pipeline(features_df, feature_cols)
# print(f"Accuracy: {results['accuracy']:.3f}, AUC: {results['auc']:.3f}")

print("✓ Pipeline function defined")
print("\nUse: results = radiomics_ml_pipeline(features_df, feature_cols)")

---

## Checklist for Students

### Preprocessing
- [ ] Handle missing/infinite values
- [ ] Standardize features (StandardScaler)
- [ ] Split data (train/test) with stratification

### Feature Selection
- [ ] Apply statistical test (F-test or mutual information)
- [ ] Select top K features
- [ ] Document selected features

### Model Training
- [ ] Use cross-validation (5-fold)
- [ ] Compare multiple models
- [ ] Tune hyperparameters (GridSearch)

### Evaluation
- [ ] Calculate accuracy, precision, recall, F1, AUC
- [ ] Generate confusion matrix
- [ ] Plot ROC and PR curves
- [ ] Analyze feature importance

### Documentation
- [ ] Save all outputs
- [ ] Document methodology
- [ ] Interpret results
- [ ] Discuss limitations